<div style="background:linear-gradient(120deg,#0B1F3A,#123A4C);color:#F6F1E7;padding:38px;border-radius:18px;">
<p style="color:#F4C95D;letter-spacing:.16em;font-size:12px;">PLAYGROUND S6E9 · OOF META LAB</p>
<h1 style="color:#F6F1E7;font-size:38px;line-height:1.12;">When Your Best Model Gets Zero Weight</h1>
<p style="font-size:19px;">L1 six-OOF is the 0.94646 control. New members must beat it on OOF.</p>
<p style="color:#00B4A6;">Den Pugovkin · do not reweight a failed pool</p></div>

### TL;DR

Rank-mean and an AUC blend of the **same six** OOF files lost **all five folds** to L1. Reweighting that pool will not beat **0.94646**. This run keeps L1 on those six as the control, then tests **new rankings**: every extra valid OOF/test pair in Input (including RealMLP and, if attached, the six-view library), after dropping Spearman ≥ 0.999 clones. A second candidate applies the published generator cliffs to the selected stack. Either recipe replaces L1 only with **+0.000020 OOF AUC and 4/5 folds**. No leaderboard anchor, no row shifts, no mix into 0.94649.

**Run:** competition + `najiama/s6e9-oof`. Optional: RealMLP output and `megayak/s6e9-six-feature-views-oof-library`. CPU. Writes `submission.csv` and `stack_report.json`.

[01 Sources](#sources) · [02 Audit](#audit) · [03 Diversity](#diversity) · [04 Honest stack](#stack) · [05 Decision](#decision)

<a id="sources"></a>
## 01 · What we borrowed — and what we rejected

| Evidence checked | Useful idea | Decision |
|---|---|---|
| [6-Model Meta-Stacking Engine](https://www.kaggle.com/code/sometimessubodh/s6e9-6-model-meta-stacking-engine) | Fold-trained L1 stack over diverse OOF files; reported 0.946349 OOF / 0.94644 public. | Rebuild the method with strict ID checks and transparent fold scores. |
| [Breaking 0.94650: Lasso Meta-Stack](https://www.kaggle.com/code/sometimessubodh/0-94651-breaking-0-94650-lasso-meta-stack) | Shows that tiny rank changes can move the public board. | Reject its external leaderboard anchor and hand-tuned row shifts. They have no OOF proof. |
| [What the 0.94650 actually is](https://www.kaggle.com/code/megayak/s6e9-what-the-0-94650-actually-is) | Six-view OOF library ~0.946345; adding it to the 0.94650 file *hurt*. Four generator cliffs worth +0.57e-5. | Attach as extra members against our 0.94646 L1. Gate the cliffs. |
| [RealMLP · PyTorch](https://www.kaggle.com/code/yekenot/ps-s6-e9-realmlp-pytorch) | Neural ranking, reported 0.94601 OOF / 0.94621 public. | Optional extra member if its output is attached. |
| [HiRGE complete OOF](https://www.kaggle.com/datasets/tamerlanomralinov/s6e9-complete-oof-npy-hirge) | Tabular neural/tree OOF on the same Stratified 5-fold seed 42. | Load only when `fold_id.npy` matches frozen v1. Skip nested maxcv. |
| Rank-mean / AUC blend of the same six | V2 log: **0/5** vs L1. | Closed. Do not rerun. |

The public OOF dataset contains derived predictions rather than extra labels. That is exactly what stacking needs: every training-row prediction was produced while that row was held out from its base learner. We still verify IDs, lengths, finiteness, and target absence before fitting anything.

In [ ]:
from pathlib import Path
import hashlib, json, os, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import HTML, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
SEED, ID, TARGET = 42, "id", "Will_Buy_EV"
NAVY, TEAL, GOLD, CORAL, PAPER = "#0B1F3A", "#00B4A6", "#F4C95D", "#E85D4C", "#F6F1E7"
STYLE = dict(template="plotly_white", font=dict(family="Inter, Arial", color=NAVY),
             title_font=dict(color=NAVY), colorway=[TEAL, GOLD, CORAL, "#526D82"])
ROOT = Path(os.environ.get("S6E9_INPUT_ROOT", "/kaggle/input" if Path("/kaggle/input").exists() else Path.cwd()))
OUTPUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs/oof-stack")
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Input root: {ROOT}\nOutput: {OUTPUT}")

In [ ]:
def one(pattern: str, roots=(ROOT,), contains: str | None = None) -> Path:
    hits = sorted({p for root in roots for p in root.rglob(pattern) if p.is_file()})
    if contains:
        filtered = [p for p in hits if contains.lower() in str(p).lower()]
        if filtered:
            hits = filtered
    if len(hits) != 1:
        raise RuntimeError(f"Expected one {pattern}; found {len(hits)}: {hits[:8]}")
    return hits[0]

train_path = one("train.csv", contains="playground-series-s6e9")
sample_path = one("sample_submission.csv", contains="playground-series-s6e9")
test_path = one("test.csv", contains="playground-series-s6e9")
train, sample, test = pd.read_csv(train_path), pd.read_csv(sample_path), pd.read_csv(test_path)
test = pd.DataFrame({ID: sample[ID]}).merge(test, on=ID, how="left", validate="one_to_one")
target_values = set(train[TARGET].dropna().unique())
y = train[TARGET].map({"No": 0, "Yes": 1}) if target_values <= {"No", "Yes"} else train[TARGET].astype(int)
assert train[ID].is_unique and sample[ID].is_unique and test[ID].equals(sample[ID]) and y.isin([0, 1]).all()
folds = np.empty(len(train), dtype=np.int8)
for fold, (_, valid) in enumerate(StratifiedKFold(5, shuffle=True, random_state=SEED).split(train, y)):
    folds[valid] = fold
fingerprint = hashlib.sha256(pd.DataFrame({ID: train[ID], "fold": folds}).to_csv(index=False).encode()).hexdigest()
print(train.shape, sample.shape, f"positive rate={y.mean():.3%}", f"fold fingerprint={fingerprint}")

<a id="audit"></a>
## 02 · Alignment is part of the model

A high AUC with shifted rows is meaningless. The loader accepts a source only when train and test IDs match the competition tables exactly after an explicit merge. It also refuses missing, duplicated, non-finite, or constant predictions. File names are mapped explicitly so a generic `submission.csv` cannot silently become the wrong member.

In [ ]:
CORE = {
    "01 Blend": ("01_blend_oof.csv", "01_submission.csv"),
    "Pure LGBM v1": ("Pure LGBM_V1_oof.csv", "Pure LGBM_V1_test.csv"),
    "Pure LGBM v3": ("Pure LGBM_V3_oof.csv", "Pure LGBM_V3_test.csv"),
    "Sergey LGBM": ("Sergey_LGBM_oof.csv", "Sergey_LGBM_submission.csv"),
    "XGB triple TE 10f": ("XGBoost_Triple_TE_10folds_oof.csv", "XGBoost_Triple_TE_10folds_test.csv"),
    "XGB triple TE 5f": ("XGBoost_Triple_TE_5folds_oof.csv", "XGBoost_Triple_TE_5folds_test.csv"),
}

def prediction(path: Path, ids: pd.Series, label: str) -> np.ndarray:
    frame = pd.read_csv(path)
    assert ID in frame and TARGET not in frame.columns[:-1], f"Unsafe schema: {path}"
    pred_cols = [c for c in frame if c != ID]
    assert len(pred_cols) == 1 and frame[ID].is_unique, f"Ambiguous schema: {path}"
    aligned = pd.DataFrame({ID: ids}).merge(frame[[ID, pred_cols[0]]], on=ID, how="left", validate="one_to_one")
    values = aligned[pred_cols[0]].to_numpy(float)
    assert len(values) == len(ids) and np.isfinite(values).all() and np.std(values) > 0
    print(f"✓ {label:<20} {path.name}")
    return values

oof, test_pred = {}, {}
for name, (oof_name, test_name) in CORE.items():
    oof[name] = prediction(one(oof_name), train[ID], name + " OOF")
    test_pred[name] = prediction(one(test_name), sample[ID], name + " test")

In [ ]:
def rank01(values: np.ndarray) -> np.ndarray:
    return pd.Series(values).rank(method="average", pct=True).to_numpy()


def buried(path: Path) -> bool:
    return any(part in {"working", "outputs"} for part in path.parts)


CLONE_RHO = 0.999
CORE_OOF = {name.lower() for name, _ in CORE.values()}
SKIP_STEMS = {"maxcv_nested", "hirge", "stack", "blend"}
consumed: set[Path] = set()


def is_clone(values: np.ndarray) -> str | None:
    ranked = pd.Series(rank01(values))
    for name, existing in oof.items():
        if float(ranked.corr(pd.Series(rank01(existing)), method="spearman")) >= CLONE_RHO:
            return name
    return None


def add_member(name: str, oof_v: np.ndarray, test_v: np.ndarray) -> bool:
    if name in oof:
        print(f"skip duplicate name {name}")
        return False
    if len(oof_v) != len(train) or len(test_v) != len(sample):
        print(f"skip {name}: length mismatch")
        return False
    if not (np.isfinite(oof_v).all() and np.isfinite(test_v).all() and np.std(oof_v) > 0):
        print(f"skip {name}: non-finite or constant")
        return False
    clone = is_clone(oof_v)
    if clone:
        print(f"skip clone {name} vs {clone}")
        return False
    oof[name] = oof_v
    test_pred[name] = test_v
    print(f"+ extra {name}")
    return True


def pred_columns(frame: pd.DataFrame) -> list[str]:
    skip = {ID, TARGET, "fold", "target", "Fold"}
    return [c for c in frame.columns if c not in skip and pd.api.types.is_numeric_dtype(frame[c])]


def aligned_col(frame: pd.DataFrame, ids: pd.Series, col: str) -> np.ndarray:
    aligned = pd.DataFrame({ID: ids}).merge(frame[[ID, col]], on=ID, how="left", validate="one_to_one")
    values = aligned[col].to_numpy(float)
    assert np.isfinite(values).all() and np.std(values) > 0
    return values

In [ ]:
def load_wide(oof_path: Path, test_path: Path, prefix: str) -> None:
    oof_df, test_df = pd.read_csv(oof_path), pd.read_csv(test_path)
    if ID not in oof_df or ID not in test_df:
        print(f"skip wide {oof_path.name}: no id")
        return
    cols = [c for c in pred_columns(oof_df) if c in test_df.columns]
    for col in cols:
        try:
            add_member(f"{prefix}/{col}", aligned_col(oof_df, train[ID], col),
                       aligned_col(test_df, sample[ID], col))
        except (AssertionError, ValueError) as exc:
            print(f"skip {prefix}/{col}: {exc}")
    consumed.update({oof_path.resolve(), test_path.resolve()})


for oof_name, test_name, prefix in (("oof_six_views.csv", "test_six_views.csv", "sixview"),):
    for op in ROOT.rglob(oof_name):
        if buried(op):
            continue
        tp = op.parent / test_name
        if tp.exists():
            load_wide(op, tp, prefix)


def partner_for(oof_path: Path) -> Path | None:
    parent = oof_path.parent
    for name in ("submission.csv", "submission_latest.csv", "test_preds.csv"):
        hit = parent / name
        if hit.exists() and hit.resolve() not in consumed:
            return hit
    stem = oof_path.stem.lower()
    for token in ("_oof", "-oof", "oof_", "oof"):
        stem = stem.replace(token, "")
    hits = []
    for path in parent.glob("*.csv"):
        name = path.name.lower()
        if path.resolve() in consumed or name == oof_path.name.lower():
            continue
        if "oof" in name or name in {"train.csv", "test.csv", "sample_submission.csv"}:
            continue
        if any(tag in name for tag in ("test", "submission", "pred")):
            hits.append(path)
    if len(hits) == 1:
        return hits[0]
    for path in hits:
        if stem and stem in path.stem.lower():
            return path
    return None


for op in sorted({p for p in ROOT.rglob("*oof*.csv") if p.is_file() and not buried(p)}):
    if op.name.lower() in CORE_OOF or op.resolve() in consumed:
        continue
    tp = partner_for(op)
    if tp is None:
        print(f"skip {op.name}: no test partner")
        continue
    try:
        add_member(op.parent.name + "/" + op.stem, prediction(op, train[ID], op.name),
                   prediction(tp, sample[ID], tp.name))
        consumed.update({op.resolve(), tp.resolve()})
    except (AssertionError, ValueError, RuntimeError) as exc:
        print(f"skip pair {op.name}: {exc}")

In [ ]:
def load_hirge() -> None:
    fold_files = [p for p in ROOT.rglob("fold_id.npy") if not buried(p)]
    if not fold_files:
        print("HiRGE skipped: attach tamerlanomralinov/s6e9-complete-oof-npy-hirge")
        return
    for fold_path in fold_files:
        fold_id = np.load(fold_path)
        if len(fold_id) != len(folds) or not np.array_equal(fold_id.astype(int), folds):
            print(f"HiRGE skip {fold_path.parent.name}: fold_id is not frozen v1")
            continue
        added = 0
        for op in sorted(fold_path.parent.glob("oof_*.npy")):
            stem = op.name[4:-4]
            if stem.lower() in SKIP_STEMS:
                print(f"skip HiRGE/{stem}: stacked/nested source")
                continue
            tp = fold_path.parent / f"test_blend_{stem}.npy"
            if not tp.exists():
                alts = [p for p in fold_path.parent.glob(f"test*{stem}*.npy") if p != op]
                tp = alts[0] if len(alts) == 1 else None
            if tp is None:
                print(f"skip HiRGE/{stem}: no test array")
                continue
            add_member(f"HiRGE/{stem}", np.load(op).astype(float), np.load(tp).astype(float))
            added += 1
        print(f"HiRGE from {fold_path.parent.name}: scanned, extras now {len(oof) - len(CORE)}")
        return

load_hirge()
EXTRA_NAMES = [name for name in oof if name not in CORE]
audit = pd.DataFrame({"Model": list(oof), "OOF AUC": [roc_auc_score(y, p) for p in oof.values()],
                      "Pool": ["core6" if n in CORE else "extra" for n in oof]})
display(audit.sort_values("OOF AUC", ascending=False).style.format({"OOF AUC": "{:.7f}"})
        .background_gradient(subset=["OOF AUC"], cmap="YlGnBu"))
print("extras:", EXTRA_NAMES or "none — attach six-view and/or HiRGE")

<a id="diversity"></a>
## 03 · Why the strongest model may receive zero weight

ROC AUC depends on ordering, so every prediction is converted to a percentile rank before stacking. The heatmap measures Spearman correlation between rankings. A model can have excellent standalone AUC and still add nothing if another member makes nearly the same errors. L1 regularization turns that redundancy into exact zero coefficients.

In [ ]:
rank_oof = pd.DataFrame({name: rank01(values) for name, values in oof.items()})
rank_test = pd.DataFrame({name: rank01(test_pred[name]) for name in rank_oof})
corr = rank_oof.corr(method="spearman")
fig = px.imshow(corr, zmin=.85, zmax=1, color_continuous_scale=[[0, CORAL], [.55, GOLD], [1, TEAL]],
                text_auto=".3f", aspect="auto", title="Prediction-rank correlation: quality is not diversity")
fig.update_layout(**STYLE, height=640)
fig.show()

<a id="stack"></a>
## 04 · Fold-honest sparse meta-model

For each outer fold the meta-model sees only the other four. The held fold is scored once. Test predictions average the five fold-wise models. `l1_core6` is frozen at `C=1` — that is the **0.94646** public file. `l1_all` is the same recipe on every extra ranking that survived the ID/clone audit. `dgp_rules` then adds the four published generator cliffs to the selected stack.

A candidate replaces its parent only with **+0.000020 OOF AUC** and **4/5** fold wins. Rank-mean and the AUC blend of the same six files already lost 0/5; they are not rerun.

In [ ]:
def store(oof_hat, test_hat, scores, weight_rows):
    assert np.isfinite(oof_hat).all() and np.isfinite(test_hat).all()
    return oof_hat, test_hat, scores, pd.DataFrame(weight_rows)


def fit_l1(columns, recipe):
    X, Xtest = rank_oof[columns].to_numpy(), rank_test[columns].to_numpy()
    oof_hat, test_hat, scores, rows = np.zeros(len(train)), np.zeros(len(sample)), [], []
    for fold in range(5):
        fit, valid = folds != fold, folds == fold
        scaler = StandardScaler().fit(X[fit])
        model = LogisticRegression(penalty="l1", solver="saga", C=1.0, max_iter=1200,
                                   tol=1e-5, random_state=SEED + fold, n_jobs=-1)
        model.fit(scaler.transform(X[fit]), y[fit])
        oof_hat[valid] = model.predict_proba(scaler.transform(X[valid]))[:, 1]
        test_hat += model.predict_proba(scaler.transform(Xtest))[:, 1] / 5
        scores.append(roc_auc_score(y[valid], oof_hat[valid]))
        rows.append(dict(recipe=recipe, fold=fold, **dict(zip(columns, model.coef_[0]))))
        print(f"{recipe:<16} fold {fold}: {scores[-1]:.7f}")
    return store(oof_hat, test_hat, scores, rows)


def rule_shift(frame: pd.DataFrame) -> np.ndarray:
    income = pd.to_numeric(frame["Annual_Income_USD"], errors="coerce").to_numpy()
    commute = pd.to_numeric(frame["Daily_Commute_km"], errors="coerce").to_numpy()
    env1 = pd.to_numeric(frame["Environmental_Concern_Level"], errors="coerce").to_numpy() == 1
    no_sub = frame["Subsidy_Available"].astype(str).to_numpy() == "No"
    anxiety = frame["Range_Anxiety_Level"].isin(["Medium", "High"]).to_numpy()
    shift = np.zeros(len(frame), dtype=float)
    shift[income >= 170537] += 10.0
    shift[(income >= 31004) & (income <= 41970)] -= 10.0
    shift[commute >= 83] -= 5.0
    shift[(income == 30000) & no_sub & (env1 | anxiety)] -= 5.0
    print("rule rows", int((shift != 0).sum()), "of", len(frame))
    return shift

In [ ]:
def py(value):
    if isinstance(value, dict):
        return {k: py(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, np.ndarray)):
        return [py(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return float(value)
    return value


def beats(parent, child, min_delta=0.000020, min_wins=4):
    parent_auc = float(roc_auc_score(y, parent[0]))
    child_auc = float(roc_auc_score(y, child[0]))
    deltas = np.asarray(child[2], float) - np.asarray(parent[2], float)
    wins = int((deltas > 0).sum())
    ok = bool((child_auc - parent_auc) > min_delta and wins >= min_wins)
    return ok, float(child_auc - parent_auc), wins, [float(x) for x in deltas], child_auc


started = time.perf_counter()
core_cols = list(CORE)
results = {"l1_core6": fit_l1(core_cols, "l1_core6")}
selected = "l1_core6"
decision = {"parent": "l1_core6", "candidates": {}, "extras": EXTRA_NAMES}
if EXTRA_NAMES:
    results["l1_all"] = fit_l1(list(oof), "l1_all")
    ok, delta, wins, deltas, child_auc = beats(results["l1_core6"], results["l1_all"])
    decision["candidates"]["l1_all"] = {"auc": child_auc, "delta": delta, "fold_wins": wins,
                                        "fold_deltas": deltas, "accepted": ok}
    if ok:
        selected = "l1_all"

shift_train, shift_test = rule_shift(train), rule_shift(test)
parent_oof, parent_test, _, _ = results[selected]
shifted_oof = parent_oof + shift_train
shifted_test = rank01(parent_test + shift_test)
rule_scores = [float(roc_auc_score(y[folds == fold], shifted_oof[folds == fold])) for fold in range(5)]
print("dgp_rules folds:", [round(s, 7) for s in rule_scores])
results["dgp_rules"] = store(shifted_oof, shifted_test, rule_scores,
                             pd.DataFrame([{"recipe": "dgp_rules", "fold": fold} for fold in range(5)]))
ok, delta, wins, deltas, child_auc = beats(results[selected], results["dgp_rules"])
decision["candidates"]["dgp_rules"] = {"auc": child_auc, "delta": delta, "fold_wins": wins,
                                       "fold_deltas": deltas, "accepted": ok, "parent": selected}
if ok:
    selected = "dgp_rules"
decision.update(selected=selected, selected_auc=float(roc_auc_score(y, results[selected][0])),
                l1_auc=float(roc_auc_score(y, results["l1_core6"][0])))
print(f"Meta training: {time.perf_counter() - started:.1f} seconds")
print(json.dumps(py(decision), indent=2))

<a id="decision"></a>
## 05 · Let the evidence choose the submission

The fold chart answers whether a new member or the generator cliffs help repeatedly. Coefficients answer a different question: which extra rankings remain useful after the original six are present? A zero L1 coefficient is a successful finding.

In [ ]:
rows = []
for recipe, (pred, _, scores, _) in results.items():
    for fold, score in enumerate(scores): rows.append({"Recipe": recipe, "Fold": fold, "AUC": score})
score_table = pd.DataFrame(rows)
fig = px.line(score_table, x="Fold", y="AUC", color="Recipe", markers=True,
              title="Same held rows, same folds: L1 six vs expanded pool vs DGP rules")
fig.update_layout(**STYLE)
fig.show()

coef = pd.concat([value[3] for name, value in results.items() if name != "dgp_rules"], ignore_index=True)
long_coef = coef.melt(id_vars=["recipe", "fold"], var_name="Model", value_name="Coefficient")
coef_summary = long_coef.groupby(["recipe", "Model"]).Coefficient.agg(["mean", "std", "min", "max"]).reset_index()
display(coef_summary.style.format({c: "{:+.4f}" for c in ["mean", "std", "min", "max"]}))

In [ ]:
for recipe, (meta_oof, meta_test, fold_scores, _) in results.items():
    pd.DataFrame({ID: train[ID], "fold": folds, "prediction": meta_oof}).to_csv(OUTPUT / f"oof_{recipe}.csv", index=False)
    pd.DataFrame({ID: sample[ID], TARGET: meta_test}).to_csv(OUTPUT / f"submission_{recipe}.csv", index=False)

chosen_oof, chosen_test = results[selected][0], results[selected][1]
if not ((chosen_test >= 0) & (chosen_test <= 1)).all():
    chosen_test = rank01(chosen_test)
submission = pd.DataFrame({ID: sample[ID], TARGET: chosen_test})
assert submission.shape == sample.shape and submission[ID].equals(sample[ID])
assert submission[TARGET].between(0, 1).all() and np.isfinite(submission[TARGET]).all()
submission.to_csv(OUTPUT / "submission.csv", index=False)
coef.to_csv(OUTPUT / "meta_coefficients.csv", index=False)
report = py({**decision, "selected_oof_auc": float(roc_auc_score(y, chosen_oof)),
          "per_fold_auc": [float(s) for s in results[selected][2]],
          "rows": {"train": int(len(train)), "test": int(len(sample))},
          "fold_fingerprint": fingerprint, "members": list(rank_oof),
          "extras": EXTRA_NAMES, "seed": SEED, "C": 1.0})
(OUTPUT / "stack_report.json").write_text(json.dumps(report, indent=2))

color = TEAL if selected != "l1_core6" else GOLD
display(HTML(f'''<div style="background:{NAVY};color:{PAPER};padding:24px;border-radius:16px;">
<p style="color:{color};font-size:13px;letter-spacing:.12em;">SELECTED RECIPE</p>
<h2 style="color:{PAPER};margin:4px 0;">{selected}</h2>
<b>OOF AUC {report['selected_oof_auc']:.7f}</b><br>
submission.csv is ready · public score is not claimed until submission
</div>'''))
print(submission.head(), "\n", sorted(p.name for p in OUTPUT.iterdir()))

### What this taught us

- **Reweighting the same six OOF files is closed.** Rank-mean and the AUC blend lost every fold to L1. Public 0.94646 stays the control until a *new ranking* beats it on OOF.
- **New members have to survive L1.** Extra OOF from the six-view library, RealMLP, or HiRGE can receive exact zero weight. That is the result, not a bug.
- **The generator cliffs are real and small.** Published gain is about +0.57e-5. They replace the stack only if they also clear +0.000020 and 4/5 on this L1 parent.
- **Do not mix into 0.94649.** Every measured dose of a weaker ranking lost a fourth decimal. `(28).csv` stays selected until public is strictly above 0.94649.

The score displayed above is computed from this run. Historical public scores are labeled as historical.